# 01. 학습 데이터 준비 (Colab)

> **AI Hub 단계(2~4)는 이제 로컬(한국 안의 Mac)에서 CLI 로 돌린다.** AI Hub 는 해외 IP 의 다운로드를 막아서
> Colab 에서는 `HTTP 502 — AI 허브는 해외에서의 데이터 다운로드를 제한하고 있습니다` 로 실패한다.
> 레포 최상위에서 `uv run python -m daboyeo_ai.data.prepare_local --out data/aihub` 를 실행하고
> (API 키는 `.env` 의 `AIHUB_APIKEY`), 만들어진 `data/aihub/parts/krid.zip`, `road.zip` 을 Drive `daboyeo_data/parts/` 에 올린다.
> Colab 에서는 **1단계(설정·코드 설치) → 5단계(RDD2022, Figshare 는 차단 없음) → 6단계(확인)** 만 실행한다.
> 2~4단계 셀은 참고용으로 남겨 두었고, CLI 와 같은 함수(`daboyeo_ai.data.selection`, `prepare_local.convert_job`)를 쓴다.

AI Hub·RDD2022 원본을 **Colab 로컬 디스크에서** 받아 우리 4클래스(`pothole, crack, guardrail, sign`) YOLO 데이터셋으로 바꾸고,
**작아진 결과만 Google Drive**에 저장한다. 원본 zip은 하나씩 처리하고 바로 지운다.

| 단계 | 셀 | 결과 |
|---|---|---|
| 1 | 설정 · Drive · 코드 설치 | |
| 2 | AI Hub 속도 시험 → **라벨만** 다운로드 | `/content/work/labels` (약 1.6 GB) |
| 3 | 라벨 구조 확인 · 개수 집계 → 받을 이미지 zip 선택 | Drive `label_counts_*.csv` |
| 4 | 이미지 zip 하나씩: 받기 → 변환 → Drive 저장 → 삭제 | Drive `parts/*.zip` |
| 5 | RDD2022 | Drive `parts/rdd2022.zip` |
| 6 | 합쳐서 확인 | 클래스·split별 개수 |

**미리 할 것**
1. AI Hub에서 두 데이터셋 다운로드 신청·승인: 「전국 도로시설물 영상정보 데이터」(71786), 「도로장애물/표면 인지 영상(수도권)」(179)
2. AI Hub에서 **API 키** 발급 → Colab 왼쪽 🔑(Secrets)에 `AIHUB_APIKEY` 로 저장, 노트북 접근 허용
3. 코드는 GitHub(`ParkGyeom/daboyeo`, 공개)에서 자동으로 받는다. 코드를 고친 뒤에는 **런타임 재시작 전에 `/content/daboyeo` 를 지우거나 `git pull`** 해야 새 코드가 반영된다
4. 데이터 준비에는 GPU가 필요 없다: **런타임 → 런타임 유형 변경 → CPU**

세션이 끊겨도 **4단계는 끝난 zip을 건너뛰고 이어서** 진행한다 (Drive `progress.json`).

In [ ]:
# ===== 설정 =====
DRIVE_ROOT = "/content/drive/MyDrive/daboyeo_data"   # 결과가 쌓이는 곳
REPO_URL = "https://github.com/ParkGyeom/daboyeo.git"  # 공개 레포라 토큰 불필요. 비우면 Drive 의 MyDrive/daboyeo 사용
REPO_DRIVE_PATH = "/content/drive/MyDrive/daboyeo"
WORK = "/content/work"                                # Colab 로컬 디스크 (임시)

KRID_KEY, ROAD_KEY = 71786, 179
# 아래 기본값은 daboyeo_ai/data/selection.py 와 같다 (로컬 CLI prepare_local 도 같은 값을 쓴다)

# 전국 도로시설물(KRID): 서울 시연과 비슷한 '도시부' + 검증셋 일부
KRID_IMAGE_ZIPS = [
    "TS_2.국도_02.도시부.zip",    # 9 GB
    "TS_3.지방도_02.도시부.zip",  # 0.7 GB
    "VS_2.국도_02.도시부.zip",    # 1 GB
    "VS_3.지방도_02.도시부.zip",  # 79 MB
]
KRID_STRIDE = 2              # 연속 프레임 중 몇 장에 1장 쓸지
KRID_INCLUDE_MEDIAN = False  # 중앙분리대도 guardrail 로 넣을지

# 도로장애물/표면: 라벨 집계 후 '포트홀 수 / GB' 가 높은 zip 부터 고른다
ROAD_TYPES = ("Mainroad", "Frontback")  # 국도·지방도, 도심 골목 (Highway·Industrialroads 제외)
ROAD_POTHOLE_ZIPS = 3
ROAD_CRACK_ZIPS = 2
ROAD_STRIDE = 5              # 1분 영상의 연속 프레임이라 5장에 1장
ROAD_NEGATIVE_RATIO = 0.1    # 보수완료 포트홀·맨홀만 있는 배경 이미지 비율

MAX_IMAGES_PER_ZIP = 4000    # zip 하나에서 쓸 최대 이미지 수 (클래스 균형용)

RDD_URL = "https://ndownloader.figshare.com/files/38030910"  # RDD2022_released_through_CRDDC2022.zip (13.3 GB)
RDD_COUNTRIES = ("Japan", "Czech", "India", "United_States", "China_MotorBike", "Norway")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
for d in (DRIVE_ROOT, f"{DRIVE_ROOT}/parts", WORK):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
!df -h /content | tail -1

### 코드 설치 (`contracts` + `ai` 패키지)

In [ ]:
import shutil, subprocess
from google.colab import userdata

REPO = "/content/daboyeo"
if not os.path.exists(REPO):
    if REPO_URL:
        url = REPO_URL
        try:
            token = userdata.get("GITHUB_TOKEN")
            url = REPO_URL.replace("https://", f"https://{token}@")
        except Exception:
            pass  # 공개 레포면 토큰 없이 받는다
        subprocess.run(["git", "clone", "-q", url, REPO], check=True)
    else:
        shutil.copytree(REPO_DRIVE_PATH, REPO, ignore=shutil.ignore_patterns(".venv", "data", "__pycache__"))
!pip install -q -e {REPO}/contracts -e {REPO}/ai

In [ ]:
import importlib, sys
sys.path[:0] = [f"{REPO}/contracts", f"{REPO}/ai"]  # 같은 세션에서 바로 import 되도록
from daboyeo_ai.data import aihub, pipeline, prepare_local, selection, sources
from daboyeo_ai.data.yolo import YoloDetWriter, CropWriter, write_data_yaml
APIKEY = userdata.get("AIHUB_APIKEY")
print("API key loaded:", bool(APIKEY))

## 2. AI Hub 파일 목록 → 속도 시험 → 라벨만 다운로드

파일 목록은 공개 API라 승인 없이도 보인다. 다운로드는 승인된 데이터셋만, **한국 IP 에서만** 된다 (Colab 은 실패 → 위 안내대로 로컬 CLI).

In [ ]:
krid_files = aihub.fetch_tree(KRID_KEY)
road_files = aihub.fetch_tree(ROAD_KEY)

krid_labels = selection.krid_label_files(krid_files)
road_labels = selection.road_label_files(road_files)
gb = lambda fs: sum(f.size_bytes for f in fs) / 1024**3
print(f"KRID 라벨 {len(krid_labels)}개 {gb(krid_labels):.2f} GB / 도로장애물 라벨 {len(road_labels)}개 {gb(road_labels):.2f} GB")

In [ ]:
# 속도 시험: 가장 작은 라벨 파일 하나. Colab(해외 서버)에서 너무 느리거나 실패하면
# 노트북/학교 서버에서 라벨과 고른 zip만 받아 전처리하고 결과만 Drive 에 올리는 방식으로 바꾼다.
import time
LABELS = pathlib.Path(WORK) / "labels"
smallest = min(krid_labels, key=lambda f: f.size_bytes)
t = time.time()
p = aihub.download(KRID_KEY, smallest, APIKEY, LABELS / "krid")
mb = p.stat().st_size / 1024**2
print(f"{smallest.name}: {mb:.1f} MB in {time.time() - t:.1f}s → {mb / max(time.time() - t, 1e-6):.1f} MB/s")

In [ ]:
from tqdm.auto import tqdm
label_paths = {}  # AihubFile.path → 로컬 zip 경로
for ds, files, sub in ((KRID_KEY, krid_labels, "krid"), (ROAD_KEY, road_labels, "road")):
    for f in tqdm(files, desc=sub):
        label_paths[f.path] = aihub.download(ds, f, APIKEY, LABELS / sub)
print(len(label_paths), "label zips ready")
!du -sh {LABELS}

## 3. 라벨 구조 확인 · 개수 집계

**도로장애물 데이터의 JSON 키 구조는 공개 문서에 없다.** 아래 출력에서 `images / annotations / categories` 가 보이고
카테고리 이름에 `포트홀`·`크랙` 등이 있으면 그대로 진행한다. 다르면 `ai/daboyeo_ai/data/sources.py` 의
`parse_coco_like` · `road_obstacle_class` 를 이 출력에 맞게 고친다 (형식이 다르면 4단계에서 바로 에러가 난다).

In [ ]:
first = lambda pred: next(label_paths[f.path] for f in road_labels if pred(f.path))
print("=== 도로장애물 TOA (포트홀) ===");  print(pipeline.peek_label_zip(first(lambda p: "/1.TOA/" in p)))
print("\n=== 도로장애물 CRACK ===");     print(pipeline.peek_label_zip(first(lambda p: "/2.CRACK/" in p)))
print("\n=== 전국 도로시설물 ===");        print(pipeline.peek_label_zip(label_paths[krid_labels[0].path], 1500))

In [ ]:
import pandas as pd
from collections import Counter

# 전국 도로시설물: 시설물 종류 × 상태(정상/파손)
krid_total = Counter()
for f in tqdm(krid_labels, desc="count KRID"):
    krid_total += pipeline.count_label_zip(label_paths[f.path], "krid")
states, krid_rows = selection.krid_count_table(krid_total)
krid_df = pd.DataFrame(krid_rows).set_index("name")
krid_df.to_csv(f"{DRIVE_ROOT}/label_counts_krid.csv")
targets = sources.KRID_SIGN | sources.KRID_GUARDRAIL | sources.KRID_MEDIAN
display(krid_df[[sources._norm(n) in targets for n in krid_df.index]])

In [ ]:
# 도로장애물: 라벨 zip 별 포트홀·크랙 수 → 대응하는 이미지 zip 크기와 함께
road_by_path = {f.path: f for f in road_files}
rows = [selection.road_count_row(f, pipeline.count_label_zip(label_paths[f.path], "coco"),
                                 selection.road_image_for(f, road_by_path), ROAD_TYPES)
        for f in tqdm(road_labels, desc="count road")]
road_df = pd.DataFrame(rows)
road_df.to_csv(f"{DRIVE_ROOT}/label_counts_road.csv", index=False)
if road_df.pothole.sum() == 0:
    raise RuntimeError("포트홀이 0개로 집계됨 — 위 '라벨 구조 확인' 출력을 보고 sources.road_obstacle_class / parse_coco_like 를 맞춰야 한다")
display(road_df.groupby(["kind", "road"])[["pothole", "crack", "negative", "image_gb"]].sum())

In [ ]:
# 받을 이미지 zip 고르기: KRID 는 정해 둔 목록, 도로장애물은 대상 도로 종류 중 '개수 / GB' 상위
picked = selection.pick_road_rows(rows, ROAD_POTHOLE_ZIPS, ROAD_CRACK_ZIPS)
jobs = selection.krid_jobs(krid_files, KRID_IMAGE_ZIPS) + selection.road_jobs(picked, road_by_path)
display(pd.DataFrame([dict(kind=j.kind, image=j.image.path, size=j.image.size) for j in jobs]))
print(f"총 {sum(j.image.size_bytes for j in jobs) / 1024**3:.1f} GB 를 하나씩 받는다")

## 4. 이미지 zip 하나씩: 받기 → 변환 → Drive 저장 → 삭제

zip 하나당 Drive `parts/<이름>.zip` 하나가 생긴다. 끝난 zip은 `progress.json` 에 기록되어 다시 실행해도 건너뛴다.

In [ ]:
import json
PROGRESS = pathlib.Path(DRIVE_ROOT) / "progress.json"
progress = json.loads(PROGRESS.read_text()) if PROGRESS.exists() else {}
settings = prepare_local.Settings(krid_stride=KRID_STRIDE, krid_include_median=KRID_INCLUDE_MEDIAN,
                                  road_stride=ROAD_STRIDE, road_negative_ratio=ROAD_NEGATIVE_RATIO,
                                  max_images_per_zip=MAX_IMAGES_PER_ZIP)

def run_job(j):
    if j.part in progress:
        print("skip (done):", j.part); return
    raw = pathlib.Path(WORK) / "raw"
    out = pathlib.Path(WORK) / "out" / j.part
    shutil.rmtree(out, ignore_errors=True)
    t = time.time()
    img_zip = aihub.download(j.ds, j.image, APIKEY, raw)
    stats = prepare_local.convert_job(j, label_paths[j.label.path], img_zip, out, settings)
    shutil.rmtree(raw, ignore_errors=True)  # 원본은 바로 지운다
    shutil.make_archive(f"{DRIVE_ROOT}/parts/{j.part}", "zip", out)
    shutil.rmtree(out, ignore_errors=True)
    progress[j.part] = dict(stats, seconds=round(time.time() - t))
    PROGRESS.write_text(json.dumps(progress, ensure_ascii=False, indent=1))
    print(j.part, dict(stats))

for j in jobs:
    run_job(j)

## 5. RDD2022 (Figshare, 신청 불필요)

In [ ]:
import json, time
PROGRESS = pathlib.Path(DRIVE_ROOT) / "progress.json"  # 4단계를 건너뛰고 이 셀만 실행해도 되도록
progress = json.loads(PROGRESS.read_text()) if PROGRESS.exists() else {}
if "rdd2022" not in progress:
    rdd_zip = pathlib.Path(WORK) / "rdd2022.zip"
    subprocess.run(["curl", "-L", "-C", "-", "-o", str(rdd_zip), RDD_URL], check=True)
    out = pathlib.Path(WORK) / "out" / "rdd2022"
    det = YoloDetWriter(out / "det")
    stats = pipeline.process_rdd2022(rdd_zip, det, RDD_COUNTRIES)
    stats.update(det.stats)
    rdd_zip.unlink()
    shutil.make_archive(f"{DRIVE_ROOT}/parts/rdd2022", "zip", out)
    shutil.rmtree(out, ignore_errors=True)
    progress["rdd2022"] = dict(stats)
    PROGRESS.write_text(json.dumps(progress, ensure_ascii=False, indent=1))
    print(dict(stats))

## 6. 합쳐서 확인

학습 노트북도 같은 방식으로 `parts/*.zip` 을 `/content/dataset` 에 풀어서 쓴다 (Drive 에서 바로 읽으면 느림).

In [ ]:
DATASET = pathlib.Path("/content/dataset")
for z in sorted(pathlib.Path(f"{DRIVE_ROOT}/parts").glob("*.zip")):
    shutil.unpack_archive(z, DATASET)
write_data_yaml(DATASET / "det")

from daboyeo_contracts import CLASS_NAMES
summary = Counter()
for txt in (DATASET / "det" / "labels").glob("*/*.txt"):
    split = txt.parent.name
    summary[(split, "images")] += 1
    for line in txt.read_text().splitlines():
        summary[(split, CLASS_NAMES[int(line.split()[0])])] += 1
det_df = pd.Series(summary).unstack(0).fillna(0).astype(int)
display(det_df)
cls_df = pd.Series(Counter(p.parent.parent.name + "/" + p.parent.name for p in (DATASET / "cls").glob("*/*/*.jpg")))
display(cls_df.sort_index())
!du -sh /content/dataset/* {DRIVE_ROOT}/parts